# Revised NFL Draft dataset description

**Research question:** Among drafted players with recorded career Approximate Value, how accurately can draft position and player characteristics distinguish late-round selections with at least one recorded Pro Bowl selection from those with none?

The original CSV was filtered for nonblank `carav` **before** the replacement 80/20 split. Zero values are retained. Run All loads only development data and the saved eligible-sample summary, never reporting records. The sample-description statistics were recorded before reservation. The split and raw source copy are already present; no preparation script is run by this notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "Data/development_set.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the nfl-draft-success-prediction repository folder.")
summary = json.loads((ROOT / "Data/eligible_sample_summary.json").read_text())
manifest = json.loads((ROOT / "Data/split_manifest.json").read_text())
df = pd.read_csv(ROOT / "Data/development_set.csv")
assert df.shape == (manifest["development_rows"], 34)
assert df["carav"].notna().all()
print(f"Loaded {len(df):,} development rows; reporting records were not opened.")
display(pd.Series({
    "Original records": summary["original_rows"],
    "Removed: blank career AV": summary["excluded_blank_carav"],
    "Eligible records": summary["eligible_rows"],
    "Development records": manifest["development_rows"],
    "Reporting records (from manifest only)": manifest["reporting_rows"],
    "Zero career AV retained in eligible sample": summary["zero_carav_retained"],
}, name="Count").to_frame())

Loaded 5,642 development rows; reporting records were not opened.


,Count
Original records,8435
Removed: blank career AV,1382
Eligible records,7053
Development records,5642
Reporting records (from manifest only),1411
Zero career AV retained in eligible sample,1067


## One-paragraph dataset description

This describes the **filtered eligible sample**, not the old 8,435-row unfiltered sample. Values come from the saved pre-reservation summary. Copy the paragraph and separate IEEE reference into the prerequisite.

In [2]:
s = summary
paragraph = (
    f"The NFL Draft 1985–2015 dataset, published by Ulrik Thyge Pedersen and downloaded from Kaggle [1], "
    f"originally contains {s['original_rows']:,} draft-selection records and {s['columns']} variables "
    "describing draft position, player characteristics, and NFL performance. Before the training/reporting "
    f"split, {s['excluded_blank_carav']:,} records with missing career Approximate Value (`carav`) were "
    f"excluded, leaving {s['eligible_rows']:,} eligible records; observed zero values were retained. "
    f"Among {s['age_observed']:,} records with age available, the mean recorded age was "
    f"{s['age_mean']:.2f} years (SD = {s['age_sample_sd']:.2f}), the median was {s['age_median']:.0f} "
    f"years (interquartile range: {s['age_q1']:.0f}–{s['age_q3']:.0f}), and ages ranged from "
    f"{s['age_min']:.0f} to {s['age_max']:.0f} years; age was missing for {s['age_missing']} records "
    f"({s['age_missing_pct']:.2f}%). The most common position groups were defensive backs "
    f"({s['positions']['DB']['pct_of_eligible']:.1f}%), linebackers "
    f"({s['positions']['LB']['pct_of_eligible']:.1f}%), and wide receivers "
    f"({s['positions']['WR']['pct_of_eligible']:.1f}%). College information was available for "
    f"{s['college_observed']:,} records ({s['college_observed_pct']:.1f}%), covering "
    f"{s['distinct_college_labels']} distinct recorded college labels. Sex, race, and ethnicity are not recorded."
)
display(Markdown(paragraph))

The NFL Draft 1985–2015 dataset, published by Ulrik Thyge Pedersen and downloaded from Kaggle [1], originally contains 8,435 draft-selection records and 34 variables describing draft position, player characteristics, and NFL performance. Before the training/reporting split, 1,382 records with missing career Approximate Value (`carav`) were excluded, leaving 7,053 eligible records; observed zero values were retained. Among 7,050 records with age available, the mean recorded age was 22.63 years (SD = 0.84), the median was 23 years (interquartile range: 22–23), and ages ranged from 20 to 29 years; age was missing for 3 records (0.04%). The most common position groups were defensive backs (19.3%), linebackers (13.6%), and wide receivers (12.2%). College information was available for 4,455 records (63.2%), covering 300 distinct recorded college labels. Sex, race, and ethnicity are not recorded.

## Reference (IEEE)

[1] U. T. Pedersen, “NFL Draft 1985–2015,” Kaggle, data set. Accessed: Nov. 17, 2026. [Online]. Available: https://www.kaggle.com/datasets/ulrikthygepedersen/nfl-draft-1985-2015

## Additional sample characteristics (saved before reservation)

Age-band percentages exclude missing age. Position percentages use all eligible records. These tables read the saved eligible-sample summary, not the reporting set.

In [3]:
display(pd.DataFrame.from_dict(summary["age_bands"], orient="index").round(1))
display(pd.DataFrame.from_dict(summary["positions"], orient="index").round(1))
display(pd.Series({
    "Age observed": summary["age_observed"], "Age missing": summary["age_missing"],
    "Mean": summary["age_mean"], "Sample SD": summary["age_sample_sd"],
    "Median": summary["age_median"], "Q1": summary["age_q1"], "Q3": summary["age_q3"],
    "Minimum": summary["age_min"], "Maximum": summary["age_max"],
}, name="Eligible sample, before reservation").to_frame().round(2))

,n,pct_of_observed_ages
20-21,500,7.1
22-23,5733,81.3
24-29,817,11.6


,n,pct_of_eligible
DB,1358,19.3
LB,961,13.6
WR,860,12.2
RB,673,9.5
DE,637,9.0
DT,537,7.6
T,497,7.0
TE,428,6.1
G,425,6.0
QB,299,4.2


,"Eligible sample, before reservation"
Age observed,7050.00
Age missing,3.00
Mean,22.63
Sample SD,0.84
Median,23.00
Q1,22.00
Q3,23.00
Minimum,20.00
Maximum,29.00


## Development-only modeling inputs

The target remains binary recorded Pro Bowl status. `carav` determined eligibility; it must not become a predictor. No model is trained here. The revision applies no additional year/round exclusion before splitting. Define rounds 4–7 as the late-round subgroup. Use grouped validation within the development partition and fit preprocessing only on training folds.

In [4]:
y = df["pb"].ge(1).astype(int).rename("success")
predictors = ["pick", "age", "position_standard", "college_univ", "tm", "year"]
X = df[predictors].copy()
groups = df["player_id"].fillna(
    "selection:" + df["year"].astype(str) + ":" + df["pick"].astype(str)
)
late_round = df["rnd"].between(4, 7)
assert "carav" not in X.columns and "pb" not in X.columns
print(f"Development inputs: {X.shape[0]:,} rows and {X.shape[1]} draft-time predictors.")
print(f"Late-round development records: {late_round.sum():,}")
print("No model fitted. Reporting data remains reserved for the final phase.")

Development inputs: 5,642 rows and 6 draft-time predictors.
Late-round development records: 2,900
No model fitted. Reporting data remains reserved for the final phase.
